In [12]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
from scipy.stats import pearsonr
sys.path.insert(0,'..')
from scripts.models.pod.model import RampPOD
from scripts.data.classes import fit_empirical_qm
warnings.filterwarnings('ignore')
pplt.rc.update({
    'tick.minor':False,
    'savefig.dpi':300,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'legend.fontsize':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})

In [13]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELSDIR  = CONFIGS['filepaths']['models']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
MODELS     = CONFIGS['experiments']
NNCONFIG   = MODELS['nn']
SRCONFIG   = MODELS['sr']
LATRANGE   = CONFIGS['domain']['latrange']
LONRANGE   = CONFIGS['domain']['lonrange']
YEARS      = CONFIGS['domain']['years']
MONTHS     = CONFIGS['domain']['months']

In [14]:
url = 'https://psl.noaa.gov/data/correlation/nina34.anom.data'
raw = pd.read_csv(url,sep=r'\s+',header=None,skiprows=1,engine='python')
raw = raw[pd.to_numeric(raw[0],errors='coerce').notna()]
raw = raw.iloc[:,:13].apply(pd.to_numeric,errors='coerce')

nino = {}
for _,row in raw.iterrows():
    year = int(row[0])
    if YEARS[0]<=year<=YEARS[-1]:
        jja = np.nanmean(row[MONTHS].to_numpy())
        if np.isfinite(jja):
            nino[year] = float(jja)

In [ ]:
SRFUNCTIONS = {
    'cube':   lambda x: x**3,
    'square': lambda x: x**2,
    'neg':    lambda x: -x,
    'sqrt':   np.sqrt,'exp':np.exp,'log':np.log,
    'abs':    np.abs,'sin':np.sin,'cos':np.cos,
}

def load_all_splits(var):
    pieces = []
    for split in ['train','valid','test']:
        path = os.path.join(SPLITSDIR,f'{split}.h5')
        with xr.open_dataset(path,engine='h5netcdf') as ds:
            if var in ds:
                pieces.append(ds[var].load())
    if not pieces:
        raise KeyError(f'{var!r} not found in any split file')
    return xr.concat(pieces,dim='time')

def flatten(a,b):
    a,b  = a.values.ravel(),b.values.ravel()
    mask = np.isfinite(a)&np.isfinite(b)
    return a[mask],b[mask]

def annual_q95(da,years):
    slices = []
    for year in years:
        mask = da.time.dt.year==year
        if not mask.any():
            continue
        q = da.sel(time=mask).quantile(0.95,dim='time',skipna=True)
        if 'quantile' in q.coords:
            q = q.drop_vars('quantile')
        slices.append(q.expand_dims(year=[year]))
    return xr.concat(slices,dim='year')

def pearson_map(q95da,ninodict):
    years     = q95da.year.values
    nino      = np.array([ninodict.get(int(year),np.nan) for year in years],dtype=float)
    q95values = q95da.values.astype(float)
    nlat,nlon = q95values.shape[1],q95values.shape[2]
    rmap      = np.full((nlat,nlon),np.nan)
    pmap      = np.full((nlat,nlon),np.nan)
    for i in range(nlat):
        for j in range(nlon):
            values = q95values[:,i,j]
            ok     = np.isfinite(values)&np.isfinite(nino)
            if ok.sum()>=5:
                rmap[i,j],pmap[i,j] = pearsonr(nino[ok],values[ok])
    coords = {'lat':q95da.lat,'lon':q95da.lon}
    return (xr.DataArray(rmap,dims=['lat','lon'],coords=coords),
            xr.DataArray(pmap,dims=['lat','lon'],coords=coords))

def _load_kernel_weights(weightsfrom):
    wlist = []
    for seed in NNCONFIG['seeds']:
        wpath = os.path.join(WEIGHTSDIR,f'{weightsfrom}_{seed}_weights.nc')
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            wlist.append(wds['k'].load())
    return xr.concat(wlist,dim='seed').mean('seed')

def _kernel_integrate_fields(fieldvars,splitvars,weightsmean):
    dsig_da = splitvars['dsig']
    if 'time' in dsig_da.dims:
        dsig_da = dsig_da.mean(dim='time')
    dsig   = np.asarray(dsig_da.transpose('sig').values).reshape(-1)
    result = {}
    for var in fieldvars:
        da        = splitvars[var]
        spacedims = [d for d in da.dims if d != 'sig']
        arr       = da.transpose(*spacedims,'sig').values
        w         = np.asarray(weightsmean.sel(field=var).values).reshape(-1)
        integ     = (arr.reshape(-1,arr.shape[-1])*w[None,:]*dsig[None,:]).sum(axis=1)
        coords    = {d:da.coords[d] for d in spacedims if d in da.coords}
        result[var] = xr.DataArray(integ.reshape(arr.shape[:-1]),dims=spacedims,coords=coords)
    return result

def load_pred_allyears(name):
    '''Load and concatenate prediction files across all splits. Returns None if any split is missing.'''
    pieces = []
    for split in ['train','valid','test']:
        fpath = os.path.join(PREDSDIR,f'{name}_{split}_predictions.nc')
        if not os.path.exists(fpath):
            return None
        with xr.open_dataset(fpath) as ds:
            pred = ds['tp'].load()
            if 'seed' in pred.dims:
                pred = pred.mean('seed')
            pieces.append(pred)
    return xr.concat(pieces,dim='time')

def eval_nn_allyears(name,runconfig):
    pred = load_pred_allyears(name)
    if pred is None:
        print(f'  {name}: prediction files not found, skipping')
    return pred

def eval_sr_allyears(name,runconfig):
    '''Load optimized SR predictions if available, otherwise evaluate inline from seed equations.'''
    pred = load_pred_allyears(name)
    if pred is not None:
        return pred
    # Inline fallback: use seed 0's best equation
    seed    = SRCONFIG['seeds'][0]
    csvpath = os.path.join(MODELSDIR,'sr',f'{name}_{seed}_equations.csv')
    if not os.path.exists(csvpath):
        print(f'  {name}: no equations file (seed {seed}), skipping')
        return None
    eqdf    = pd.read_csv(csvpath)
    bestrow = eqdf.loc[eqdf['loss'].idxmin()]
    eqstr   = str(bestrow['equation'])
    compl   = int(bestrow['complexity'])
    print(f'  {name} inline (seed {seed}, complexity={compl}): {eqstr}')
    with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
        srstats = json.load(f)
    zmin      = (0.0-srstats['tp_mean'])/srstats['tp_std']
    fieldvars = runconfig['fieldvars']
    localvars = runconfig.get('localvars',[])
    weightsfrom = runconfig.get('weightsfrom')
    pieces = []
    for split in ['train','valid','test']:
        normpath = os.path.join(SPLITSDIR,f'norm_{split}.h5')
        with xr.open_dataset(normpath,engine='h5netcdf') as ds:
            svars = {v:ds[v].load() for v in ds.data_vars}
        refda = svars['tp']
        ns    = dict(SRFUNCTIONS,__builtins__={})
        if weightsfrom:
            weights = _load_kernel_weights(weightsfrom)
            for var,da in _kernel_integrate_fields(fieldvars,svars,weights).items():
                ns[var] = da.transpose('time','lat','lon').values.ravel()
        else:
            for var in fieldvars:
                da = svars[var]
                extra = [d for d in da.dims if d not in ('time','lat','lon')]
                ns[var] = (da.mean(dim=extra) if extra else da).transpose('time','lat','lon').values.ravel()
        for var in localvars:
            da = svars.get(var)
            if da is None:
                continue
            extra = [d for d in da.dims if d not in ('time','lat','lon')]
            ns[var] = da.broadcast_like(refda).transpose('time','lat','lon').values.ravel()
        ypredn  = np.asarray(eval(eqstr,ns),dtype=float)
        if ypredn.ndim==0:
            ypredn = np.full(refda.values.ravel().shape,float(ypredn))
        ypredmm = np.maximum(np.expm1(np.maximum(ypredn,zmin)*srstats['tp_std']+srstats['tp_mean']),0.0)
        pieces.append(xr.DataArray(ypredmm.reshape(refda.shape),dims=refda.dims,coords=refda.coords))
    return xr.concat(pieces,dim='time') if pieces else None

In [16]:
trainds = xr.open_dataset(os.path.join(SPLITSDIR,'train.h5'),engine='h5netcdf')[['tp','pr']].load()
validds = xr.open_dataset(os.path.join(SPLITSDIR,'valid.h5'),engine='h5netcdf')[['tp','pr']].load()
era5    = xr.concat([trainds.tp,validds.tp],dim='time')
imerg   = xr.concat([trainds.pr,validds.pr],dim='time')
era5,imerg = xr.align(era5,imerg,join='inner')
era5,imerg = flatten(era5,imerg)
qmfunction = fit_empirical_qm(era5,imerg)
del trainds,validds,era5,imerg

In [17]:
imergall = load_all_splits('pr')
q95imerg = annual_q95(imergall,YEARS)
rimerg,pimerg = pearson_map(q95imerg,nino)
del imergall,q95imerg

In [18]:
bl = load_all_splits('bl')
with np.load(os.path.join(MODELSDIR,'pod','pod_bl.npz')) as d:
    pod = RampPOD(float(d['alpha']),float(d['xcrit']))
predpod = xr.DataArray(pod.forward(bl).reshape(bl.shape),dims=bl.dims,coords=bl.coords)
qmpod   = xr.apply_ufunc(qmfunction,predpod,vectorize=False)
q95pod  = annual_q95(qmpod,YEARS)
rpod,ppod = pearson_map(q95pod,nino)
del bl,predpod,qmpod,q95pod

In [ ]:
base = eval_nn_allyears('base_all',NNCONFIG['runs']['base_all'])
if base is not None:
    qmbase  = xr.apply_ufunc(qmfunction,base,vectorize=False)
    q95base = annual_q95(qmbase,YEARS)
    rbase,pbase = pearson_map(q95base,nino)
    del base,qmbase,q95base
else:
    rbase,pbase = None,None

In [ ]:
kernel = eval_nn_allyears('gauss_all',NNCONFIG['runs']['gauss_all'])
if kernel is not None:
    qmkernel  = xr.apply_ufunc(qmfunction,kernel,vectorize=False)
    q95kernel = annual_q95(qmkernel,YEARS)
    rkernel,pkernel = pearson_map(q95kernel,nino)
    del kernel,qmkernel,q95kernel
else:
    rkernel,pkernel = None,None

In [ ]:
srresults = {}
for _name,_rc in SRCONFIG['runs'].items():
    srresults[_name] = eval_sr_allyears(_name,_rc)
for _name,_eqspec in SRCONFIG.get('optimizedeqs',{}).items():
    _rc = SRCONFIG['runs'][_eqspec['runfrom']]
    srresults[_name] = eval_sr_allyears(_name,_rc)

rsr,psr = None,None
for srname,srpred in srresults.items():
    if srpred is not None:
        q95sr = annual_q95(xr.apply_ufunc(qmfunction,srpred,vectorize=False),YEARS)
        rsr,psr = pearson_map(q95sr,nino)
        break  # use the first available SR result for the summary panel

In [ ]:
rsr,psr = None,None
for srname in SRCONFIG.get('optimizedeqs',{}):
    if srname in srresults and srresults[srname] is not None:
        q95 = annual_q95(xr.apply_ufunc(qmfunction,srresults[srname],vectorize=False),YEARS)
        rsr,psr = pearson_map(q95,nino)
        break

PANELS = [
    (rimerg, pimerg, 'IMERG V06'),
    (rpod,   ppod,   'POD'),
    (rbase,  pbase,  'Baseline NN'),
    (rkernel,pkernel,'Parametric Kernel NN'),
    (rsr,    psr,    'Optimized SR'),
]

fig,axs = pplt.subplots(ncols=5,proj='cyl',refwidth=2)
axs.format(coast=True,borders=True,latlim=(LATRANGE[0],LATRANGE[1]),lonlim=(LONRANGE[0],LONRANGE[1]),latlines=5,lonlines=5)
mappable = None
for ax,(r,p,title) in zip(axs,PANELS):
    ax.format(title=title)
    if r is None:
        continue
    m = ax.pcolormesh(r.lon.values,r.lat.values,r.values,cmap='ColdHot',vmin=-0.7,vmax=0.7,N=15)
    if mappable is None:
        mappable = m
    if p is not None:
        pmask = p.values < 0.05
        if pmask.any():
            lons2d,lats2d = np.meshgrid(r.lon.values,r.lat.values)
            ax.scatter(lons2d[pmask],lats2d[pmask],color='k',s=1,zorder=5,linewidths=0)
fig.colorbar(mappable,loc='b',label='Pearson $r$  (JJA NINO3.4 Anomalies vs. JJA 95th-percentile Precipitation)')
# fig.save('../figs/fig_enso.jpg')
pplt.show()

In [ ]:
sr_opt_names  = list(SRCONFIG.get('optimizedeqs',{}).keys())
available_opt = [n for n in sr_opt_names if n in srresults and srresults[n] is not None]

if not available_opt:
    print('No optimized SR predictions available.')
else:
    sr_pearson = {}
    for name in available_opt:
        q95       = annual_q95(xr.apply_ufunc(qmfunction,srresults[name],vectorize=False),YEARS)
        sr_pearson[name] = pearson_map(q95,nino)

    ncols   = 1+len(available_opt)
    fig,axs = pplt.subplots(ncols=ncols,proj='cyl',refwidth=2)
    axs.format(coast=True,borders=True,latlim=(LATRANGE[0],LATRANGE[1]),lonlim=(LONRANGE[0],LONRANGE[1]),latlines=5,lonlines=5)
    mappable = None

    axs[0].format(title='IMERG V06')
    m = axs[0].pcolormesh(rimerg.lon.values,rimerg.lat.values,rimerg.values,cmap='ColdHot',vmin=-0.7,vmax=0.7,N=15)
    mappable = m
    if pimerg is not None:
        pmask = pimerg.values < 0.05
        if pmask.any():
            lons2d,lats2d = np.meshgrid(rimerg.lon.values,rimerg.lat.values)
            axs[0].scatter(lons2d[pmask],lats2d[pmask],c='k',s=1,zorder=5,linewidths=0)

    for ci,name in enumerate(available_opt):
        ax  = axs[ci+1]
        r,p = sr_pearson[name]
        ax.format(title=name.replace('_',' '))
        m = ax.pcolormesh(r.lon.values,r.lat.values,r.values,cmap='ColdHot',vmin=-0.7,vmax=0.7,N=15)
        if p is not None:
            pmask = p.values < 0.05
            if pmask.any():
                lons2d,lats2d = np.meshgrid(r.lon.values,r.lat.values)
                ax.scatter(lons2d[pmask],lats2d[pmask],c='k',s=1,zorder=5,linewidths=0)

    fig.colorbar(mappable,loc='b',label='Pearson $r$  (JJA NINO3.4 Anomalies vs. JJA 95th-percentile Precipitation)')
    # fig.save('../figs/fig_enso_sr.jpg')
    pplt.show()